In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
os.chdir('/content/drive/Othercomputers/My Laptop/Work/Grievance Redressal Project')

In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ==========================================
# SETUP & MOCK DATA (Updated for timezone-aware dates)
# ==========================================
historical_data = pd.DataFrame({
    'complaint_id': [101, 102, 103, 104],
    '_id': ['C001', 'C002', 'C003', 'C001'],
    'state': ['Odisha', 'Odisha', 'Delhi', 'Odisha'],

    # Using the specific timezone-aware datetime format
    'recvd_date': pd.to_datetime([
        '2026-03-15 10:30:00.000000+00:00',
        '2026-04-10 08:15:22.123000+00:00',
        '2026-04-12 14:45:10.000000+00:00',
        '2026-04-14 00:01:45.593000+00:00'
    ]),
    'grievance_text_processed': [
        "water supply pipe broken near main road",
        "electricity fluctuation in sector 4",
        "potholes on the highway causing accidents",
        "water supply pipe broken near main road"
    ]
})

new_grievance = {
    '_id': 'C001',
    'state': 'Odisha',
    'recvd_date': pd.to_datetime('2026-04-15 09:00:00.000000+00:00'),
    'grievance_text_processed': "broken water supply pipe on main road"
}

##  BLOCKING

In [6]:
def apply_blocking(new_record, db_df, days_window=30):
    # pd.Timedelta handles timezone-aware math seamlessly
    lower_bound = new_record['recvd_date'] - pd.Timedelta(days=days_window)
    upper_bound = new_record['recvd_date']

    time_condition = (
        (db_df['recvd_date'] >= lower_bound) &
        (db_df['recvd_date'] <= upper_bound)
    )

    entity_condition = (
        (db_df['state'] == new_record['state']) &
        (db_df['_id'] == new_record['_id'])
    )

    filtered_db = db_df[time_condition & entity_condition].copy()
    filtered_db.reset_index(drop=True, inplace=True)
    return filtered_db

print("Applying Blocking Phase...")
search_space_df = apply_blocking(new_grievance, historical_data)

Applying Blocking Phase...


## TF-IDF VECTORIZATION & SIMILARITY

In [7]:
if not search_space_df.empty:
    print(f"Applying TF-IDF against {len(search_space_df)} historical records...")

    vectorizer = TfidfVectorizer(stop_words='english')

    historical_texts = search_space_df['grievance_text_processed'].tolist()
    new_text = [new_grievance['grievance_text_processed']]

    all_texts = historical_texts + new_text

    tfidf_matrix = vectorizer.fit_transform(all_texts)

    historical_vectors = tfidf_matrix[:-1]
    new_vector = tfidf_matrix[-1]

    similarities = cosine_similarity(new_vector, historical_vectors).flatten()

    top_k = min(3, len(similarities))
    top_indices = similarities.argsort()[-top_k:][::-1]

    print("\n--- RESULTS ---")
    for i, idx in enumerate(top_indices):
        match_score = similarities[idx]
        matched_record = search_space_df.iloc[idx]

        status = "Unique"
        if match_score >= 0.85: status = "EXACT/HIGH DUPLICATE"
        elif match_score >= 0.60: status = "POTENTIAL DUPLICATE"

        print(f"Rank {i+1}: {status} ({match_score:.4f}) -> {matched_record['grievance_text_processed']}")

else:
    print("No historical records found.")

Applying TF-IDF against 1 historical records...

--- RESULTS ---
Rank 1: EXACT/HIGH DUPLICATE (0.8674) -> water supply pipe broken near main road


## Zero Shot Classification

In [8]:
# Install if needed
# !pip install transformers pandas torch

import pandas as pd
from transformers import pipeline

# ==========================================
# SAMPLE DATA
# ==========================================
data = pd.DataFrame({
    'complaint_id': [201, 202, 203, 204],
    'grievance_text_processed': [
        "the main water pipe is leaking heavily on the street causing flooding",
        "bad",
        "!!!@@@ PLEASE FIX THIS NOW $$$$$ !!!!!!",
        "call 9876543210 to win free lottery 12345"
    ]
})

# ==========================================
# LOAD MODEL
# ==========================================
print("Loading model...")

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0  # use -1 if no GPU
)

# ==========================================
# LABELS
# ==========================================
candidate_labels = [
    "valid public service complaint or grievance",
    "spam, advertisement, or scam message",
    "abusive, offensive, or threatening language",
    "irrelevant or meaningless text"
]

# ==========================================
# EXTRA FEATURES (optional but useful)
# ==========================================
def extra_features(text):
    return {
        "length": len(text.split()),
        "has_number": any(char.isdigit() for char in text),
        "has_url": "http" in text or "www" in text
    }

# ==========================================
# PROCESSING LOOP
# ==========================================
results = []

for _, row in data.iterrows():
    text = str(row['grievance_text_processed'])

    # Zero-shot classification
    result = classifier(text, candidate_labels, multi_label=True)
    scores = dict(zip(result['labels'], result['scores']))

    # Extract scores
    spam = scores.get("spam, advertisement, or scam message", 0)
    abuse = scores.get("abusive, offensive, or threatening language", 0)
    irrelevant = scores.get("irrelevant or meaningless text", 0)
    valid = scores.get("valid public service complaint or grievance", 0)

    # Extra features
    features = extra_features(text)

    # ==========================================
    # DECISION LOGIC
    # ==========================================
    if spam > 0.80 or (features["has_number"] and spam > 0.60):
        action = "❌ REJECTED (Spam)"
    elif irrelevant > 0.75 or features["length"] <= 2:
        action = "❌ REJECTED (Irrelevant)"
    elif abuse > 0.85 and valid < 0.50:
        action = "❌ REJECTED (Abusive)"
    elif abuse > 0.60:
        action = "⚠️ FLAGGED (Abusive but valid)"
    else:
        action = "✅ ACCEPTED"

    results.append({
        "complaint_id": row["complaint_id"],
        "text": text,
        "spam_score": round(spam, 3),
        "abuse_score": round(abuse, 3),
        "irrelevant_score": round(irrelevant, 3),
        "valid_score": round(valid, 3),
        "action": action
    })

# ==========================================
# DISPLAY RESULTS
# ==========================================
results_df = pd.DataFrame(results)

for _, row in results_df.iterrows():
    print(f"[{row['action']}]")
    print(f"Text: {row['text']}")
    print(f"Scores → Valid: {row['valid_score']}, Spam: {row['spam_score']}, Abuse: {row['abuse_score']}, Irrelevant: {row['irrelevant_score']}")
    print("-" * 80)

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[✅ ACCEPTED]
Text: the main water pipe is leaking heavily on the street causing flooding
Scores → Valid: 0.936, Spam: 0.0, Abuse: 0.062, Irrelevant: 0.0
--------------------------------------------------------------------------------
[❌ REJECTED (Irrelevant)]
Text: bad
Scores → Valid: 0.05, Spam: 0.665, Abuse: 0.965, Irrelevant: 0.059
--------------------------------------------------------------------------------
[⚠️ FLAGGED (Abusive but valid)]
Text: !!!@@@ PLEASE FIX THIS NOW $$$$$ !!!!!!
Scores → Valid: 0.917, Spam: 0.017, Abuse: 0.644, Irrelevant: 0.0
--------------------------------------------------------------------------------
[✅ ACCEPTED]
Text: call 9876543210 to win free lottery 12345
Scores → Valid: 0.064, Spam: 0.154, Abuse: 0.017, Irrelevant: 0.001
--------------------------------------------------------------------------------
